# Colab gpu kernel playground

This notebook writes a tiny CUDA kernel, compiles it, and runs it on the Colab GPU.


In [5]:
%%writefile custom_kernel.cu
#include <cstdlib>
#include <iostream>
#include <cuda_runtime.h>

#define TILE_WIDTH 16

__global__ void tiledMatMul(float* M, float* N, float* P, int Width) {
    __shared__ float Mds[TILE_WIDTH][TILE_WIDTH];
    __shared__ float Nds[TILE_WIDTH][TILE_WIDTH];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Identify the row and column of the P element to work on
    int Row = by * TILE_WIDTH + ty;
    int Col = bx * TILE_WIDTH + tx;

    // Loop over the M and N tiles required to compute P element
    float Pvalue = 0.0f;
    for (int ph = 0; ph < Width / TILE_WIDTH; ++ph) {
        // Collaborative loading of M and N tiles into shared memory
        Mds[ty][tx] = M[Row * Width + ph * TILE_WIDTH + tx];
        Nds[ty][tx] = N[(ph * TILE_WIDTH + ty) * Width + Col];
        __syncthreads();

        for (int k = 0; k < TILE_WIDTH; ++k) {
            Pvalue += Mds[ty][k] * Nds[k][tx];
        }

        __syncthreads();
    }

    P[Row * Width + Col] = Pvalue;
}

int main() {
    const int Width = 16;
    const int size = Width * Width;

    float* hM = new float[size];
    float* hN = new float[size];
    float* hP = new float[size];

    std::srand(1234);
    for (int row = 0; row < Width; ++row) {
        for (int col = 0; col < Width; ++col) {
            hM[row * Width + col] = static_cast<float>(std::rand() % 10);
            hN[row * Width + col] = (row == col) ? 1.0f : 0.0f;
        }
    }

    float* dM;
    float* dN;
    float* dP;

    cudaMalloc((void**)&dM, size * sizeof(float));
    cudaMalloc((void**)&dN, size * sizeof(float));
    cudaMalloc((void**)&dP, size * sizeof(float));

    cudaMemcpy(dM, hM, size * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(dN, hN, size * sizeof(float), cudaMemcpyHostToDevice);

    dim3 dimBlock(TILE_WIDTH, TILE_WIDTH);
    dim3 dimGrid(Width / TILE_WIDTH, Width / TILE_WIDTH);

    tiledMatMul<<<dimGrid, dimBlock>>>(dM, dN, dP, Width);
    cudaDeviceSynchronize();

    cudaMemcpy(hP, dP, size * sizeof(float), cudaMemcpyDeviceToHost);

    std::cout << "Random matrix M:" << std::endl;
    for (int row = 0; row < Width; ++row) {
        for (int col = 0; col < Width; ++col) {
            std::cout << hM[row * Width + col] << "\t";
        }
        std::cout << std::endl;
    }

    std::cout << std::endl;
    std::cout << "Identity matrix N:" << std::endl;
    for (int row = 0; row < Width; ++row) {
        for (int col = 0; col < Width; ++col) {
            std::cout << hN[row * Width + col] << "\t";
        }
        std::cout << std::endl;
    }

    std::cout << std::endl;
    std::cout << "Matrix P = M x N:" << std::endl;
    for (int row = 0; row < Width; ++row) {
        for (int col = 0; col < Width; ++col) {
            std::cout << hP[row * Width + col] << "\t";
        }
        std::cout << std::endl;
    }

    bool matches = true;
    for (int i = 0; i < size; ++i) {
        if (static_cast<int>(hM[i]) != static_cast<int>(hP[i])) {
            matches = false;
            break;
        }
    }

    std::cout << std::endl;
    if (matches) {
        std::cout << "Check passed: hM and hP match." << std::endl;
    } else {
        std::cout << "Check failed: hM and hP do not match." << std::endl;
    }

    cudaFree(dM);
    cudaFree(dN);
    cudaFree(dP);
    delete[] hM;
    delete[] hN;
    delete[] hP;

    return 0;
}


Overwriting custom_kernel.cu


In [6]:
!nvidia-smi

Mon May  4 22:21:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             51W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
!nvcc custom_kernel.cu -o custom_kernel

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [8]:
!./custom_kernel

Random matrix M:
4	9	5	7	0	1	5	8	2	3	5	1	6	3	8	1	
8	5	4	1	0	4	8	9	5	0	4	2	6	6	2	0	
5	9	0	7	0	5	5	4	0	1	6	8	4	6	9	2	
3	6	4	4	0	4	5	7	6	1	1	3	7	4	5	4	
5	7	2	6	4	7	2	7	8	0	5	5	8	5	9	2	
1	3	8	3	0	5	0	6	8	1	1	7	7	9	1	3	
6	3	1	3	3	5	0	3	6	5	8	6	0	8	8	3	
3	6	6	5	3	8	2	1	0	5	8	9	4	0	2	3	
5	5	6	0	1	6	4	9	3	2	5	6	2	4	9	8	
2	8	3	6	6	5	7	8	3	8	8	7	8	2	2	5	
0	0	6	1	6	0	0	2	4	7	8	9	3	7	7	6	
7	0	4	4	6	3	4	1	1	4	8	9	9	3	7	9	
3	3	0	2	5	2	4	1	1	2	0	5	1	7	1	9	
8	7	5	6	0	9	7	2	6	7	3	5	2	0	4	6	
5	6	0	0	8	4	2	9	8	2	4	1	2	7	0	0	
4	7	8	5	9	7	9	5	6	2	2	9	5	6	5	0	

Identity matrix N:
1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	0	
0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	0	
0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	0	
0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	0	
0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	0	
0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	0	
0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	0	
0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	0	
0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	0	
0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	0	
0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	0	
0	0	0	0	0	0	0	0	0	0	0	0	1	0	0	0	
0	0	0	

In [5]:
!ncu ./custom_kernel

==PROF== Connected to process 16600 (/content/custom_kernel)
==PROF== Profiling "matMul" - 0: 0%....50%....100% - 10 passes
Matrix A (2x3):
1	2	3	
4	5	6	

Matrix B (3x4):
7	8	9	10	
11	12	13	14	
15	16	17	18	

Matrix C = A x B (2x4):
74	80	86	92	
173	188	203	218	
==PROF== Disconnected from process 16600
[16600] custom_kernel@127.0.0.1
  matMul(float *, float *, float *, int, int, int) (2, 1, 1)x(2, 2, 1), Context 1, Stream 7, Device 0, CC 8.0
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         1.19
    SM Frequency                    Ghz         1.08
    Elapsed Cycles                cycle        3,942
    Memory Throughput                 %         0.47
    DRAM Throughput                   %         0.11
    Duration                         us         3.65
    L1/TEX Cache Throughput        